# Sistema de Visão Computacional para Segurança do Trabalho

Pipeline completo: detecção de EPIs (YOLOv8n) + segmentação de pessoas (YOLOv8n-seg), avaliação no conjunto de teste e inferência em vídeo, incluindo o relatório de conformidade.

Ver `docs/relatorio-tecnico.md` no repositório para o relatório completo com todos os resultados já obtidos.

**Como usar este notebook:** rode as células em ordem. Cada célula corresponde a um script `src/stepNN_*.py` do repositório (mesmo código, mesmos hiperparâmetros documentados).


## 0. Setup do ambiente

In [ ]:
# Clonar o repositório (ajuste a URL para o seu fork/repositório no GitHub)
# !git clone https://github.com/<seu-usuario>/<seu-repositorio>.git
# %cd <seu-repositorio>

# Se preferir, faça upload do projeto via Google Drive e ajuste o %cd abaixo:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/<caminho-do-projeto>

!pip install -q -r requirements.txt


### Token do Kaggle

O `step01` baixa o dataset "Construction Site Safety" do Kaggle e exige autenticação.
Gere um token em [kaggle.com/settings/api](https://www.kaggle.com/settings/api) e configure-o
(opção A recomendada no Colab: usar `Secrets` do Colab; opção B: colar direto, célula abaixo).

In [ ]:
import os
from pathlib import Path

# Opção B (simples, mas exponha o token só nesta sessão - não commite!):
KAGGLE_TOKEN = ""  # cole aqui o token gerado em kaggle.com/settings/api

if KAGGLE_TOKEN:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "access_token").write_text(KAGGLE_TOKEN)
    print("Token do Kaggle configurado.")
else:
    print("Configure KAGGLE_TOKEN acima antes de rodar o step01.")


In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available())
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 1. Fase 1 — Baixar e amostrar o dataset de detecção (Construction Site Safety)

Baixa o dataset completo do Kaggle (requer token, célula acima) e seleciona um subconjunto de ~350 imagens por amostragem estratificada (seed=42), garantindo presença mínima de todas as 10 classes de EPI.

In [ ]:
%run src/step01_data_prepare_construction_site_safety.py

## 2. Fase 1 — Baixar e amostrar o dataset de segmentação (COCO person)

Baixa as anotações COCO 2017 (val2017), filtra a categoria `person` com máscara, e baixa 300 imagens (seed=42). Não exige credenciais (fonte pública).

In [ ]:
%run src/step02_data_prepare_coco_person_subset.py

## 3. Fase 1 — Gerar splits reprodutíveis (70/20/10)

Gera as listas de treino/validação/teste para as duas fontes, com seed=42.

In [ ]:
%run src/step03_data_make_splits.py

## 4. Fase 1 — Análise exploratória (EDA)

Contagem de instâncias por classe, resolução e brilho médio (proxy de iluminação) do dataset de detecção. Gráficos salvos em `reports/figures/`.

In [ ]:
%run src/step04_data_eda_construction_site_safety.py

## 5. Fase 2 — Preparar configuração YOLO (detecção)

Gera `data.yaml` e as listas `train.txt`/`val.txt`/`test.txt` no formato Ultralytics, a partir dos splits da Fase 1.

In [ ]:
%run src/step05_detection_prepare_yolo_config.py

## 6. Fase 2 — Treinar o detector (YOLOv8n)

Fine-tuning de 30 épocas, imgsz=640, batch=16, seed=42 (hiperparâmetros documentados em `docs/relatorio-tecnico.md`). **Com GPU (Colab), isso roda em poucos minutos** — no desenvolvimento original (CPU), levou ~59 minutos.

In [ ]:
%run src/step06_detection_train_yolo.py

## 7. Fase 3 — Converter COCO para YOLO-seg (segmentação)

Converte os polígonos de anotação do COCO para o formato YOLO-seg, reaproveitando os splits da Fase 1.

In [ ]:
%run src/step07_segmentation_prepare_yolo_config.py

## 8. Fase 3 — Treinar o segmentador (YOLOv8n-seg)

Fine-tuning de 30 épocas, mesmos hiperparâmetros de forma da Fase 2, classe única `person`.

In [ ]:
%run src/step08_segmentation_train_yolo.py

## 9. Fase 3 — Comparação visual caixas × máscaras

Roda os dois modelos treinados sobre as mesmas imagens do domínio de canteiro de obra, gerando exemplos comentados em `reports/figures/`.

In [ ]:
%run src/step09_segmentation_compare_boxes_vs_masks.py

## 10. Fase 4 — Concatenar o vídeo final

Concatena os clipes baixados em `video/input/` em um único vídeo ≥30s (padroniza resolução e fps). Requer os clipes originais já em `video/input/` (ver `video/input/README.md` para as fontes).

In [ ]:
%run src/step10_inference_concat_videos.py

## 11. Fase 4 — Avaliação no conjunto de teste

Roda `model.val(split="test")` para os dois modelos: mAP@0.5, mAP@0.5:0.95, precisão, recall, matriz de confusão, e IoU médio (detector).

In [ ]:
%run src/step11_evaluation_test_set.py

## 12. Fase 4 — Análise de erros

Seleciona exemplos de falso positivo/negativo do detector no teste, por maior discrepância de contagem por classe, com visualizações comentadas.

In [ ]:
%run src/step12_evaluation_error_analysis.py

## 13. Fase 4 — Inferência em vídeo

Roda os dois modelos sobre o vídeo final, salvando as saídas anotadas em `video/output/`.

In [ ]:
%run src/step13_inference_run_on_video.py

## 14. Aplicação prática — Relatório de conformidade de EPI

Agrega as detecções de violação (`NO-Hardhat`, `NO-Mask`, `NO-Safety Vest`) em eventos (início/fim/duração/confiança), salvando `reports/violacoes-epi.parquet` e um resumo em Markdown.

In [ ]:
%run src/step14_inference_compliance_report.py

## Resultados

Ver [`docs/relatorio-tecnico.md`](../docs/relatorio-tecnico.md) no repositório para o relatório
técnico completo com todas as métricas, análise de erros, comparação caixas×máscaras e o
relatório de conformidade de EPI obtidos ao rodar este pipeline.
